<a href="https://colab.research.google.com/github/AndresCanaveralVergara/PT_Iris_ACV/blob/master/PT_Iris_ACV_VF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import (
    col, when, sum as _sum, month, to_date, count as _count,
    regexp_replace, upper, trim, explode, abs as _abs, datediff,
    row_number, broadcast
)

RUTA_CSV = "/content/transacciones.csv"
RUTA_TRM = "/content/trm.json"
RUTA_ANULADAS = "/content/anuladas.txt"

spark = SparkSession.builder.appName("reporte_transacciones").getOrCreate()

# Leer transacciones
df = (spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(RUTA_CSV))

# Conteo por mes (bucle infinito corregido -> una sola agregación)
conteo_por_mes = (df
    .withColumn("mes", month(to_date(col("fecha"))))
    .groupBy("mes")
    .agg(_count("*").alias("transacciones"))
    .orderBy("mes"))
conteo_por_mes.show()

# Parsear (convertir a cadena de texto) fecha como date
df = df.withColumn("fecha_dt", to_date(col("fecha")))

# Limpieza de monto (quitar $ y separador de miles antes de castear)
df = df.withColumn("monto_num",
        regexp_replace(col("monto"), r"[\$\.]", "").cast("double"))

# Normalización de moneda
df = df.withColumn("moneda_norm", upper(trim(col("moneda"))))
df = df.withColumn("moneda_norm",
        when(col("moneda_norm") == "DOLARES", "USD")
        .otherwise(col("moneda_norm")))

# Normalización de segmento (PYME es sigla, no usar initcap)
segmento_upper = upper(trim(col("segmento")))
df = df.withColumn("segmento_norm",
        when(segmento_upper == "PREMIUM", "Premium")
        .when(segmento_upper == "PERSONAL", "Personal")
        .when(segmento_upper == "PYME", "PYME")
        .when(segmento_upper == "EMPRESARIAL", "Empresarial")
        .otherwise(None))

# Quitar transacciones duplicadas
df = df.dropDuplicates(["id_transaccion"])

# 2. Cargar TRM real (JSON anidado)
trm_raw = spark.read.option("multiLine", True).json(RUTA_TRM)
trm_df = (trm_raw
    .select(explode(col("tasas")).alias("t"))
    .select(
        to_date(col("t.fecha")).alias("fecha_trm"),
        col("t.trm").cast("double").alias("trm")
    ))

# Emparejar cada transacción USD con la TRM más cercana disponible
# Criterio: se prioriza la TRM vigente en o antes de la fecha de la transacción
# TRM del día/día hábil anterior. Si la transacción es anterior a la primera TRM registrada (no hay ninguna TRM previa disponible), se usa la TRM más próxima que exista, como mejor aproximación posible.
usd_tx = df.filter(col("moneda_norm") == "USD").select("id_transaccion", "fecha_dt")

candidatos = (usd_tx.alias("tx")
    .crossJoin(broadcast(trm_df).alias("trm"))
    .withColumn("diff_dias", datediff(col("tx.fecha_dt"), col("trm.fecha_trm")))
    .withColumn("prioridad", when(col("diff_dias") >= 0, 0).otherwise(1))
    .withColumn("brecha", _abs(col("diff_dias"))))

w = Window.partitionBy("tx.id_transaccion").orderBy(col("prioridad"), col("brecha"))

trm_asignada = (candidatos
    .withColumn("rn", row_number().over(w))
    .filter(col("rn") == 1)
    .select(col("tx.id_transaccion").alias("id_transaccion"),
            col("trm.trm").alias("trm_aplicada")))

df = df.join(trm_asignada, on="id_transaccion", how="left")

# 3. Excluir transacciones anuladas
anuladas_df = (spark.read
    .option("header", True)
    .option("sep", "|")
    .option("comment", "#")
    .csv(RUTA_ANULADAS))

df = df.join(broadcast(anuladas_df.select("id_transaccion")),
              on="id_transaccion", how="left_anti")

# 4. Calcular monto en COP
df = df.withColumn("monto_cop",
        when(col("moneda_norm") == "USD", col("monto_num") * col("trm_aplicada"))
        .when(col("moneda_norm") == "COP", col("monto_num"))
        .otherwise(None))

df.select("id_transaccion", "moneda_norm", "monto_num", "trm_aplicada",
          "monto_cop", "segmento_norm").orderBy("id_transaccion").show(20, truncate=False)

# 5. Total transado en COP por segmento (una sola agregación, ya no un loop)
print("=== Total transado en COP por segmento ===")
(df.groupBy("segmento_norm")
   .agg(_sum("monto_cop").alias("total_cop"), _count("*").alias("num_transacciones"))
   .orderBy("segmento_norm")
   .show(truncate=False))

spark.stop()

+---+-------------+
|mes|transacciones|
+---+-------------+
|  1|            6|
|  2|            4|
|  3|            8|
+---+-------------+

+--------------+-----------+---------+------------+---------+-------------+
|id_transaccion|moneda_norm|monto_num|trm_aplicada|monto_cop|segmento_norm|
+--------------+-----------+---------+------------+---------+-------------+
|1             |COP        |1200000.0|NULL        |1200000.0|Premium      |
|2             |COP        |850000.0 |NULL        |850000.0 |Personal     |
|3             |USD        |2000.0   |3900.0      |7800000.0|PYME         |
|5             |COP        |3500000.0|NULL        |3500000.0|Premium      |
|6             |COP        |1000000.0|NULL        |1000000.0|Personal     |
|7             |COP        |500000.0 |NULL        |500000.0 |Personal     |
|8             |USD        |1500.0   |4050.0      |6075000.0|PYME         |
|9             |COP        |1.2E7    |NULL        |1.2E7    |Empresarial  |
|10            |USD    